# Nool - Enterprise Knowledge Assistant: Answer Evaluation

This notebook checks Qwen's answers to the 162 initial-state questions using a Colab GPU. It uses the existing company search index and does not change the 50,000-passage search benchmark.

1. In Colab, choose **Runtime > Change runtime type > T4 GPU**.
2. Put `company_search_artifact.zip` in `MyDrive/Nool/`.
3. Run each cell from top to bottom. Keep the notebook open until the answer check finishes.
4. Download `company_answer_predictions.jsonl` from `MyDrive/Nool/` and share it for review.

The notebook uses Ollama's local API inside the Colab runtime. It needs no API key. The Colab runtime and its Ollama installation are temporary.


## Prepare the project and search index

This cell checks the GPU, connects Google Drive, gets the project code, loads the existing index ZIP, and prepares the original fictional company database. Approve the Drive access prompt when Colab asks.


In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab, then run this cell again.")

drive.mount("/content/drive")
repo = Path("/content/Nool")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/nayanojwal0810/Nool.git", str(repo)],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pyyaml>=6.0.1",
        "bm25s==0.3.16",
        "PyStemmer==3.1.0",
        "pypdf>=5.0,<7.0",
        "transformers==5.22.1",
    ],
    check=True,
)

artifact = Path("/content/drive/MyDrive/Nool/company_search_artifact.zip")
if not artifact.is_file():
    raise FileNotFoundError(f"Put the existing company search ZIP at {artifact}")
with zipfile.ZipFile(artifact) as archive:
    archive.extractall(repo / "indexes")

project_env = os.environ.copy()
project_env["PYTHONPATH"] = str(repo / "src")
subprocess.run(
    [sys.executable, "scripts/ingest_demo_documents.py"],
    cwd=repo,
    env=project_env,
    check=True,
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Company search index and original demo documents are ready.")

## Start the local answer model

This cell downloads the official Ollama Linux archive inside Colab, starts its local API, and downloads `qwen2.5:3b`. Wait until it prints that the model is ready. No Ollama account or key is needed.


In [ ]:
import time
from urllib.request import urlopen

archive_path = "/tmp/ollama-linux-amd64.tar.zst"
subprocess.run(
    [
        "curl",
        "-fL",
        "-o",
        archive_path,
        "https://ollama.com/download/ollama-linux-amd64.tar.zst",
    ],
    check=True,
)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
subprocess.run(["tar", "-C", "/usr", "-xf", archive_path], check=True)
log = open("/content/ollama_server.log", "w", encoding="utf-8")
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log,
    stderr=subprocess.STDOUT,
    env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
)
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError("Ollama stopped. Check /content/ollama_server.log")
    try:
        with urlopen("http://127.0.0.1:11434/api/tags", timeout=2):
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start. Check /content/ollama_server.log")
subprocess.run(["ollama", "pull", "qwen2.5:3b"], check=True)
subprocess.run(
    ["ollama", "run", "qwen2.5:3b", "Reply OK only."],
    check=True,
    stdout=subprocess.PIPE,
    text=True,
)
placement = subprocess.check_output(["ollama", "ps"], text=True)
if "GPU" not in placement:
    raise RuntimeError("Ollama is using CPU. Select a GPU runtime and restart.")
print("Local Ollama API and Qwen model are ready on GPU.")

## Check all prepared answers

This cell runs every initial-state case. It prints automatic counts and saves each answer with the expected source for human review. The counts are not an answer-accuracy score.


In [ ]:
output = repo / "outputs/company_answer_predictions.jsonl"
subprocess.run(
    [
        sys.executable,
        "scripts/evaluate_company_answers.py",
        "--device",
        "cuda",
        "--output",
        str(output),
    ],
    cwd=repo,
    env=project_env,
    check=True,
)
destination = Path("/content/drive/MyDrive/Nool/company_answer_predictions.jsonl")
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(output, destination)
print(f"Saved review file: {destination}")

## Review the results

Download `company_answer_predictions.jsonl` from Drive and add it to the project's `outputs/` folder on your computer. Check whether answers are correct, whether each statement follows from its cited excerpt, and whether restricted or unsupported questions are refused. The four document-update and four permission-change cases are tested separately in the project tests.
